# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2024

playin_losers = (  # remove play-in losers from seeding data
    3357,  # Sacred Heart
    3162,  # Columbia
    3120,  # Auburn
    3221,  # Holy Cross
)

model_path = '../data/models/womens/2025_03_15_model.pkl'
data_path = '../data/models/womens/2025_03_15_data.parquet'

season

2024

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\womens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2024,3101,Abilene Chr,-1.0,-1.0
1,2024,3102,Air Force,-1.0,-1.0
2,2024,3103,Akron,-1.0,-1.0
3,2024,3104,Alabama,0.0,0.0
4,2024,3105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0
374,2024,3477,East Texas A&M,-1.0,-1.0
375,2024,3478,Le Moyne,-1.0,-1.0
376,2024,3479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

Omitted

In [3]:
# df_barttorvik = pd.read_parquet(r'..\data\preprocessed\womens_barttorvik\barttorvik.parquet')

# df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

# df_barttorvik

In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\WTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 3192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,3394
1,a&m-corpus christi,3394
2,abilene chr,3101
3,abilene christian,3101
4,abilene-christian,3101
...,...,...
1171,youngstown st.,3464
1172,youngstown state,3464
1173,youngstown-st,3464
1174,youngstown-state,3464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1170

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_19372\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

# df_match.head(25)

In [8]:
# df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

# df_barttorvik

In [9]:
# df = pd.merge(
#     df,
#     df_barttorvik.drop(columns=['TEAM']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

### Past Seasons

In [10]:
df_ps = pd.read_parquet(r'..\data\preprocessed\womens_past_seasons\past_seasons_ratings.parquet')

df_ps = df_ps.loc[df_ps['Season'] == season, :].reset_index(drop=True)

df_ps

,Season,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,Abilene Christian,-0.002761,-0.036218
1,2024,Air Force,-0.019225,-0.021754
2,2024,Akron,-0.040582,-0.020406
3,2024,Alabama,0.274209,0.217993
4,2024,Alabama A&M,-0.168682,-0.098402
...,...,...,...,...
358,2024,Wright State,-0.173424,-0.053027
359,2024,Wyoming,0.098008,0.069632
360,2024,Xavier,-0.059519,-0.060891
361,2024,Yale,-0.035396,0.026920


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ps['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Abilene Christian,abilene christian,100
4,Quinnipiac,quinnipiac,100
5,Queens (NC),queens (nc),100
6,Purdue Fort Wayne,purdue fort wayne,100
7,Purdue,purdue,100
8,Providence,providence,100
9,Princeton,princeton,100


In [12]:
df_ps.insert(1, 'TeamID', df_ps['Team'].map(team_to_spelling).map(spelling_to_id))

df_ps

,Season,TeamID,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,3101,Abilene Christian,-0.002761,-0.036218
1,2024,3102,Air Force,-0.019225,-0.021754
2,2024,3103,Akron,-0.040582,-0.020406
3,2024,3104,Alabama,0.274209,0.217993
4,2024,3105,Alabama A&M,-0.168682,-0.098402
...,...,...,...,...,...
358,2024,3460,Wright State,-0.173424,-0.053027
359,2024,3461,Wyoming,0.098008,0.069632
360,2024,3462,Xavier,-0.059519,-0.060891
361,2024,3463,Yale,-0.035396,0.026920


In [13]:
df = pd.merge(
    df,
    df_ps.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,3101,Abilene Chr,-1.0,-1.0,-0.002761,-0.036218
1,2024,3102,Air Force,-1.0,-1.0,-0.019225,-0.021754
2,2024,3103,Akron,-1.0,-1.0,-0.040582,-0.020406
3,2024,3104,Alabama,0.0,0.0,0.274209,0.217993
4,2024,3105,Alabama A&M,-1.0,-1.0,-0.168682,-0.098402
...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN
374,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN
375,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN


In [14]:
df.loc[df['Past Year Efficiency Margin'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN
225,2024,3327,Okla City,-1.0,-1.0,NaN,NaN


In [15]:
df.loc[df['Past Year Efficiency Margin'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin


### My Rankings

In [16]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\womens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,South Carolina,5.348664,0.540655,1.199302,0.658648,74.409140
1,2024,Connecticut,3.913262,0.504011,1.189835,0.685824,72.866365
2,2024,Texas,3.870221,0.456575,1.177144,0.720569,71.739541
3,2024,UCLA,3.817370,0.426926,1.138986,0.712061,71.853088
4,2024,Southern California,3.746376,0.373978,1.127198,0.753220,70.859265
...,...,...,...,...,...,...,...
355,2024,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
356,2024,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
357,2024,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
358,2024,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [17]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Le Moyne,le moyne,100
2,Elon,elon,100
3,Loyola (MD),loyola (md),100
4,UC San Diego,uc san diego,100
5,Yale,yale,100
6,Idaho,idaho,100
7,Howard,howard,100
8,Iona,iona,100
9,Youngstown State,youngstown state,100


In [18]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,3376,South Carolina,5.348664,0.540655,1.199302,0.658648,74.409140
1,2024,3163,Connecticut,3.913262,0.504011,1.189835,0.685824,72.866365
2,2024,3400,Texas,3.870221,0.456575,1.177144,0.720569,71.739541
3,2024,3417,UCLA,3.817370,0.426926,1.138986,0.712061,71.853088
4,2024,3425,Southern California,3.746376,0.373978,1.127198,0.753220,70.859265
...,...,...,...,...,...,...,...,...
355,2024,3354,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
356,2024,3447,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
357,2024,3254,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
358,2024,3476,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [19]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [20]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,3101,Abilene Chr,-1.0,-1.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008
1,2024,3102,Air Force,-1.0,-1.0,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056
2,2024,3103,Akron,-1.0,-1.0,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028
3,2024,3104,Alabama,0.0,0.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116
4,2024,3105,Alabama A&M,-1.0,-1.0,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643
...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844
374,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300
375,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

Omitted

In [22]:
# df_starters = pd.read_parquet(fr'..\data\preprocessed\womens_starters\starters_{season}.parquet')

# df_starters.insert(0, 'Season', season)

# df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

# df_starters

In [23]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

# df_match.head(25)

In [24]:
# df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

# df_starters

In [25]:
# df_starters.loc[df_starters['TeamID'].isna(), :]

In [26]:
# df = pd.merge(
#     df,
#     df_starters.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [27]:
# df.loc[df['Starters'].isna(), :]

### Openskill Ratings

In [28]:
df_os = pd.read_parquet(fr'..\data\preprocessed\womens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2024,South Carolina,59.034668,45.727392
1,2024,Texas,53.074235,40.494205
2,2024,Connecticut,51.460005,39.055079
3,2024,Iowa,51.402284,38.731977
4,2024,Southern California,51.110974,38.512240
...,...,...,...,...
355,2024,Alabama State,1.718860,-12.127996
356,2024,McNeese State,2.277615,-12.622216
357,2024,Houston Christian,1.520705,-12.900479
358,2024,South Carolina State,-0.539184,-14.171589


In [29]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Delaware,delaware,100
2,Quinnipiac,quinnipiac,100
3,Wichita State,wichita state,100
4,Loyola Marymount,loyola marymount,100
5,Abilene Christian,abilene christian,100
6,St. Thomas,st. thomas,100
7,New Mexico State,new mexico state,100
8,Eastern Illinois,eastern illinois,100
9,Radford,radford,100


In [30]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2024,3376,South Carolina,59.034668,45.727392
1,2024,3400,Texas,53.074235,40.494205
2,2024,3163,Connecticut,51.460005,39.055079
3,2024,3234,Iowa,51.402284,38.731977
4,2024,3425,Southern California,51.110974,38.512240
...,...,...,...,...,...
355,2024,3106,Alabama State,1.718860,-12.127996
356,2024,3270,McNeese State,2.277615,-12.622216
357,2024,3223,Houston Christian,1.520705,-12.900479
358,2024,3354,South Carolina State,-0.539184,-14.171589


In [31]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
0,2024,3101,Abilene Chr,-1.0,-1.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008,18.501742,5.651852
1,2024,3102,Air Force,-1.0,-1.0,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056,20.312118,7.604695
2,2024,3103,Akron,-1.0,-1.0,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028,17.960257,4.853112
3,2024,3104,Alabama,0.0,0.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116,41.428647,29.490924
4,2024,3105,Alabama A&M,-1.0,-1.0,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643,17.055286,4.652751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,2.006163,-10.928439
374,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,20.249666,7.487434
375,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,23.057138,10.392918
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Standard Stats

Omitted

In [33]:
# df_ss = pd.read_parquet('../data/preprocessed/womens_standard_stats/standard_stats.parquet')

# df_ss = df_ss.loc[df_ss['Season'] == season, :].reset_index(drop=True)

# df_ss

In [34]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ss['Team'].unique())

# df_match.head(25)

In [35]:
# df_ss.insert(1, 'TeamID', df_ss['Team'].map(team_to_spelling).map(spelling_to_id))

# df_ss

In [36]:
# df = pd.merge(
#     df,
#     df_ss.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [37]:
# df.loc[df['Team Win%'].isna(), :]

### Map to Matchups

In [38]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [39]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

# df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2024,1,W,False,3376
1,2024,2,W,False,3323
2,2024,3,W,False,3333
3,2024,4,W,False,3231
4,2024,5,W,False,3328
...,...,...,...,...,...
63,2024,12,Z,True,3435
64,2024,13,Z,False,3267
65,2024,14,Z,False,3238
66,2024,15,Z,False,3263


In [40]:
df = df.merge(
    df_seeds,
    how='left',
    on=['Season', 'TeamID'],
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Seed,Region,Play In
0,2024,3101,Abilene Chr,-1.0,-1.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008,18.501742,5.651852,NaN,NaN,NaN
1,2024,3102,Air Force,-1.0,-1.0,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056,20.312118,7.604695,NaN,NaN,NaN
2,2024,3103,Akron,-1.0,-1.0,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028,17.960257,4.853112,NaN,NaN,NaN
3,2024,3104,Alabama,0.0,0.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116,41.428647,29.490924,8.0,X,False
4,2024,3105,Alabama A&M,-1.0,-1.0,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643,17.055286,4.652751,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,2.006163,-10.928439,NaN,NaN,NaN
374,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,20.249666,7.487434,NaN,NaN,NaN
375,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,23.057138,10.392918,NaN,NaN,NaN
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Remap to Team A / Team B format

In [41]:
id_to_region = dict(zip(df['TeamID'], df['Region']))
id_to_seed = dict(zip(df['TeamID'], df['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df['TeamID'].unique() 
        for team_b in df['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2024,3101,3102,NaN,NaN,NaN,NaN
1,2024,3101,3103,NaN,NaN,NaN,NaN
2,2024,3101,3104,NaN,X,NaN,8.0
3,2024,3101,3105,NaN,NaN,NaN,NaN
4,2024,3101,3106,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
142501,2024,3480,3475,NaN,NaN,NaN,NaN
142502,2024,3480,3476,NaN,NaN,NaN,NaN
142503,2024,3480,3477,NaN,NaN,NaN,NaN
142504,2024,3480,3478,NaN,NaN,NaN,NaN


Get round of matchup

In [42]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # ignore play-in games

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0         False
1         False
2         False
3         False
4         False
          ...  
142501    False
142502    False
142503    False
142504    False
142505    False
Length: 142506, dtype: bool

In [43]:
df_mod['Round'] = float('nan')

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod['Round'].describe()

count    4548.000000
mean        5.095866
std         1.190665
min         1.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: Round, dtype: float64

Get home court advantage

In [44]:
df_mod['Location'] = 0

# conditions: after 2012 but not 2021 (covid stadium), matchup is within first 2 rounds, and team is top 4 seed
df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team A Seed'] <= 4), 
    'Location'
] = 1

df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team B Seed'] <= 4), 
    'Location'
] = -1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location
0,2024,3101,3102,NaN,NaN,NaN,NaN,NaN,0
1,2024,3101,3103,NaN,NaN,NaN,NaN,NaN,0
2,2024,3101,3104,NaN,X,NaN,8.0,NaN,0
3,2024,3101,3105,NaN,NaN,NaN,NaN,NaN,0
4,2024,3101,3106,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
142501,2024,3480,3475,NaN,NaN,NaN,NaN,NaN,0
142502,2024,3480,3476,NaN,NaN,NaN,NaN,NaN,0
142503,2024,3480,3477,NaN,NaN,NaN,NaN,NaN,0
142504,2024,3480,3478,NaN,NaN,NaN,NaN,NaN,0


In [45]:
df_mod['Seed'] = df_mod['Team A Seed'] - df_mod['Team B Seed']

# df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed
0,2024,3101,3102,NaN,NaN,NaN,NaN,NaN,0,NaN
1,2024,3101,3103,NaN,NaN,NaN,NaN,NaN,0,NaN
2,2024,3101,3104,NaN,X,NaN,8.0,NaN,0,NaN
3,2024,3101,3105,NaN,NaN,NaN,NaN,NaN,0,NaN
4,2024,3101,3106,NaN,NaN,NaN,NaN,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...
142501,2024,3480,3475,NaN,NaN,NaN,NaN,NaN,0,NaN
142502,2024,3480,3476,NaN,NaN,NaN,NaN,NaN,0,NaN
142503,2024,3480,3477,NaN,NaN,NaN,NaN,NaN,0,NaN
142504,2024,3480,3478,NaN,NaN,NaN,NaN,NaN,0,NaN


Get Head-to-Head

In [46]:
df_h2h = pd.read_parquet('../data/preprocessed/womens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2024,Abilene Christian,Alabama,NaN,-1.242485
1,2024,Abilene Christian,Albany (NY),NaN,0.000000
2,2024,Abilene Christian,Alcorn State,NaN,-0.308721
3,2024,Abilene Christian,American,NaN,0.882921
4,2024,Abilene Christian,Arizona,NaN,-0.701317
...,...,...,...,...,...
66407,2024,Youngstown State,Wichita State,NaN,-0.081696
66408,2024,Youngstown State,Wisconsin,NaN,-0.729834
66409,2024,Youngstown State,Wright State,0.023094,-0.172887
66410,2024,Youngstown State,Wyoming,NaN,-0.684239


In [47]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Queens (NC),queens (nc),100
2,Purdue Fort Wayne,purdue fort wayne,100
3,Purdue,purdue,100
4,Providence,providence,100
5,Princeton,princeton,100
6,Presbyterian,presbyterian,100
7,Prairie View,prairie view,100
8,Portland State,portland state,100
9,Quinnipiac,quinnipiac,100


In [48]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2024,3101,Abilene Christian,3104,Alabama,NaN,-1.242485
1,2024,3101,Abilene Christian,3107,Albany (NY),NaN,0.000000
2,2024,3101,Abilene Christian,3108,Alcorn State,NaN,-0.308721
3,2024,3101,Abilene Christian,3110,American,NaN,0.882921
4,2024,3101,Abilene Christian,3112,Arizona,NaN,-0.701317
...,...,...,...,...,...,...,...
66407,2024,3464,Youngstown State,3455,Wichita State,NaN,-0.081696
66408,2024,3464,Youngstown State,3458,Wisconsin,NaN,-0.729834
66409,2024,3464,Youngstown State,3460,Wright State,0.023094,-0.172887
66410,2024,3464,Youngstown State,3461,Wyoming,NaN,-0.684239


In [49]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2024,3101,3102,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
1,2024,3101,3103,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,2024,3101,3104,NaN,X,NaN,8.0,NaN,0,NaN,NaN,-1.242485
3,2024,3101,3105,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
4,2024,3101,3106,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2024,3480,3475,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142502,2024,3480,3476,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142503,2024,3480,3477,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142504,2024,3480,3478,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN


Get team names

In [50]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\WTeams.csv')

df_teams

,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M
...,...,...
373,3476,Stonehill
374,3477,East Texas A&M
375,3478,Le Moyne
376,3479,Mercyhurst


In [51]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2024,3101,Abilene Chr,3102,Air Force,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
1,2024,3101,Abilene Chr,3103,Akron,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,2024,3101,Abilene Chr,3104,Alabama,NaN,X,NaN,8.0,NaN,0,NaN,NaN,-1.242485
3,2024,3101,Abilene Chr,3105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
4,2024,3101,Abilene Chr,3106,Alabama St,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2024,3480,West Georgia,3475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142502,2024,3480,West Georgia,3476,Stonehill,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142503,2024,3480,West Georgia,3477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142504,2024,3480,West Georgia,3478,Le Moyne,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN


Map features

In [52]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

# df_features['Team A ADJOE Team B ADJDE'] = team_a_features['ADJOE'] + team_b_features['ADJDE']
# df_features['Team B ADJOE Team A ADJDE'] = team_b_features['ADJOE'] + team_a_features['ADJDE']

# df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
# df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A Efficiency Margin'] = team_a_features['Efficiency Margin']
df_features['Team B Efficiency Margin'] = team_b_features['Efficiency Margin']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,0.0,0.0,0.016465,-0.014464,-0.289313,0.015779,0.050619,0.034840,-2.069048,-1.810375,-1.952843,-0.030567,-0.046346
1,0.0,0.0,0.037821,-0.015812,0.246509,0.081456,0.078902,-0.002554,1.008980,0.541485,0.798740,-0.030567,-0.112023
2,-1.0,-1.0,-0.276969,-0.254211,-3.226946,-0.290821,-0.128381,0.162440,-1.660108,-22.926905,-23.839072,-0.030567,0.260254
3,0.0,0.0,0.165921,0.062184,0.597221,0.102807,0.116437,0.013630,-0.201635,1.446456,0.999101,-0.030567,-0.133374
4,0.0,0.0,0.159452,0.104764,1.877140,0.292048,0.234794,-0.057254,-1.163349,16.782882,17.779848,-0.030567,-0.322615
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030908
142502,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.371073
142503,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.136922
142504,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.134376


In [53]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,2024,3101,Abilene Chr,3102,Air Force,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,0.016465,-0.014464,-0.289313,0.015779,0.050619,0.034840,-2.069048,-1.810375,-1.952843,-0.030567,-0.046346
1,2024,3101,Abilene Chr,3103,Akron,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,0.037821,-0.015812,0.246509,0.081456,0.078902,-0.002554,1.008980,0.541485,0.798740,-0.030567,-0.112023
2,2024,3101,Abilene Chr,3104,Alabama,NaN,X,NaN,8.0,NaN,0,NaN,NaN,-1.242485,-1.0,-1.0,-0.276969,-0.254211,-3.226946,-0.290821,-0.128381,0.162440,-1.660108,-22.926905,-23.839072,-0.030567,0.260254
3,2024,3101,Abilene Chr,3105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,0.165921,0.062184,0.597221,0.102807,0.116437,0.013630,-0.201635,1.446456,0.999101,-0.030567,-0.133374
4,2024,3101,Abilene Chr,3106,Alabama St,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,0.159452,0.104764,1.877140,0.292048,0.234794,-0.057254,-1.163349,16.782882,17.779848,-0.030567,-0.322615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2024,3480,West Georgia,3475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030908
142502,2024,3480,West Georgia,3476,Stonehill,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.371073
142503,2024,3480,West Georgia,3477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.136922
142504,2024,3480,West Georgia,3478,Le Moyne,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.134376


In [54]:
# track if game is a tournament matchup for later
tournament_matchup = (df_mod['Team A Region'].notna()) & (df_mod['Team B Region'].notna())

tournament_matchup.sum()

4556

In [55]:
df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,2024,3101,Abilene Chr,3102,Air Force,NaN,0,NaN,NaN,NaN,0.0,0.0,0.016465,-0.014464,-0.289313,0.015779,0.050619,0.034840,-2.069048,-1.810375,-1.952843,-0.030567,-0.046346
1,2024,3101,Abilene Chr,3103,Akron,NaN,0,NaN,NaN,NaN,0.0,0.0,0.037821,-0.015812,0.246509,0.081456,0.078902,-0.002554,1.008980,0.541485,0.798740,-0.030567,-0.112023
2,2024,3101,Abilene Chr,3104,Alabama,NaN,0,NaN,NaN,-1.242485,-1.0,-1.0,-0.276969,-0.254211,-3.226946,-0.290821,-0.128381,0.162440,-1.660108,-22.926905,-23.839072,-0.030567,0.260254
3,2024,3101,Abilene Chr,3105,Alabama A&M,NaN,0,NaN,NaN,NaN,0.0,0.0,0.165921,0.062184,0.597221,0.102807,0.116437,0.013630,-0.201635,1.446456,0.999101,-0.030567,-0.133374
4,2024,3101,Abilene Chr,3106,Alabama St,NaN,0,NaN,NaN,NaN,0.0,0.0,0.159452,0.104764,1.877140,0.292048,0.234794,-0.057254,-1.163349,16.782882,17.779848,-0.030567,-0.322615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2024,3480,West Georgia,3475,Southern Indiana,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030908
142502,2024,3480,West Georgia,3476,Stonehill,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.371073
142503,2024,3480,West Georgia,3477,East Texas A&M,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.136922
142504,2024,3480,West Georgia,3478,Le Moyne,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.134376


Drop features that were not used in the model

In [56]:
df_mod.drop(
    columns=[
        'Round',
        'Mu',
        'Adjusted Tempo',
        'Location',
        'Common Opps',
    ],
    inplace=True,
)

Check that data follows same format as the data that the model was trained on

In [57]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [58]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.778518753537347,
               lambda_l1=2.096041209255663, lambda_l2=0.24121410842083418,
               learning_rate=0.025864767729102435, max_depth=8, metric='rmse',
               min_child_samples=3,
               monotone_constraints=[-1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1],
               n_estimators=1000, num_leaves=155, random_state=22,
               verbosity=-1)

In [59]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

df_mod['Prediction'] = mod.predict_proba(X)[:, 1]

df_mod['Prediction']

0         0.649523
1         0.715397
2         0.061645
3         0.717396
4         0.930093
            ...   
142501    0.336601
142502    0.336601
142503    0.336601
142504    0.336601
142505    0.336601
Name: Prediction, Length: 142506, dtype: float64

In [60]:
(
    df_mod[['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B', 'Prediction']]
    .to_parquet(f'../data/simulations/womens/matchup_predictions_{season}.parquet')
)

'Done'

'Done'

Turn predictions into matchup matrix

In [61]:
# filter down to just tournament games
df_mod = df_mod.loc[tournament_matchup, :].reset_index(drop=True)

# filter out play-in losers
df_mod = df_mod.loc[
    (~df_mod['Team A ID'].isin(playin_losers)) & (~df_mod['Team B ID'].isin(playin_losers)), 
    :
].reset_index(drop=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Seed,Head to Head,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin,Prediction
0,2024,3104,Alabama,3112,Arizona,-3.0,NaN,-1.0,-2.333333,-0.011964,-0.088412,-0.106160,-0.004288,0.019550,0.023838,2.636996,0.260254,0.264542,0.464361
1,2024,3104,Alabama,3124,Baylor,3.0,NaN,-1.0,-1.666667,-0.003632,-0.184383,-0.466910,-0.076243,0.003907,0.080150,-1.937833,0.260254,0.336497,0.284176
2,2024,3104,Alabama,3151,Chattanooga,-6.0,NaN,0.0,0.666667,0.260483,0.305787,1.358484,0.181149,0.092727,-0.088422,2.525677,0.260254,0.079105,0.872115
3,2024,3104,Alabama,3160,Colorado,3.0,NaN,-2.0,-0.333333,-0.046413,-0.008156,-0.989598,-0.090021,-0.050877,0.039144,0.680708,0.260254,0.350275,0.263652
4,2024,3104,Alabama,3163,Connecticut,5.0,NaN,-2.0,-3.666667,-0.156755,-0.244462,-1.522447,-0.243756,-0.126163,0.117594,-9.564155,0.260254,0.504011,0.059661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,3465,Cal Baptist,3428,Utah,10.0,NaN,-3.0,-1.666667,-0.332888,-0.194443,-2.763110,-0.293504,-0.170015,0.123489,-2.995926,0.059370,0.352873,0.049412
4028,2024,3465,Cal Baptist,3435,Vanderbilt,3.0,NaN,0.0,0.000000,-0.057153,-0.077703,-1.705536,-0.140465,-0.032123,0.108342,1.155616,0.059370,0.199835,0.165999
4029,2024,3465,Cal Baptist,3439,Virginia Tech,11.0,NaN,-5.0,-2.666667,-0.351406,-0.277246,-2.530129,-0.266728,-0.164393,0.102335,-5.090748,0.059370,0.326098,0.049412
4030,2024,3465,Cal Baptist,3452,West Virginia,7.0,NaN,-1.0,-1.000000,-0.153552,-0.179322,-2.245023,-0.274592,-0.093751,0.180841,-2.574384,0.059370,0.333962,0.067290


In [62]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID', 'Prediction']]
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,3104,3112,3124,3151,3160,3163,3166,3179,3180,3181,3186,3193,3195,3199,3211,3231,3234,3235,3238,3242,3243,3245,3257,3261,3263,3266,3267,3268,3276,3277,3279,3292,3301,3304,3313,3314,3323,3326,3328,3333,3339,3342,3343,3349,3350,3355,3376,3390,3393,3394,3397,3400,3401,3404,3414,3417,3424,3425,3428,3435,3439,3452,3453,3465
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3104,NaN,0.464361,0.284176,0.872115,0.263652,0.059661,0.251614,0.731626,0.939002,0.377849,0.776080,0.737840,0.720819,0.722212,0.144686,0.226917,0.137388,0.297087,0.870120,0.528151,0.372310,0.872463,0.453128,0.135520,0.872892,0.705978,0.846284,0.295208,0.629402,0.272700,0.243216,0.685219,0.245352,0.334641,0.871724,0.682506,0.227643,0.227916,0.436957,0.268284,0.871659,0.939055,0.333246,0.859669,0.685231,0.727081,0.042903,0.120129,0.386821,0.939055,0.181534,0.120505,0.829489,0.939002,0.871620,0.132429,0.424698,0.133570,0.234763,0.855036,0.246814,0.281530,0.610610,0.871759
3112,0.522571,NaN,0.267489,0.847610,0.213671,0.060299,0.346719,0.726952,0.934227,0.494232,0.845713,0.737094,0.614547,0.630604,0.158258,0.233352,0.149048,0.323161,0.871833,0.588827,0.452987,0.871669,0.251191,0.211457,0.873541,0.703333,0.787576,0.299431,0.714680,0.364824,0.354099,0.608364,0.240150,0.407208,0.872995,0.674202,0.255122,0.173633,0.412189,0.184859,0.870120,0.939055,0.315787,0.870807,0.655863,0.726513,0.046241,0.094121,0.616908,0.933700,0.296615,0.063569,0.707178,0.939055,0.872866,0.070368,0.330899,0.177490,0.515349,0.762757,0.246755,0.405455,0.658308,0.847610
3124,0.714035,0.732431,NaN,0.952046,0.510266,0.233476,0.601510,0.764878,0.954425,0.713842,0.947969,0.846558,0.763309,0.761441,0.440570,0.295652,0.334055,0.398347,0.952477,0.554387,0.438702,0.954425,0.607347,0.406126,0.952046,0.765060,0.871395,0.632906,0.744336,0.619730,0.753734,0.720774,0.290954,0.698982,0.954425,0.734705,0.419855,0.372071,0.524189,0.500162,0.939034,0.954425,0.627557,0.954425,0.678179,0.764325,0.133366,0.239718,0.702704,0.954425,0.610417,0.207089,0.767530,0.954425,0.934285,0.325262,0.647834,0.423429,0.682581,0.772434,0.501746,0.775772,0.812979,0.952046
3151,0.123920,0.146705,0.049412,NaN,0.049412,0.046241,0.080969,0.248213,0.549876,0.066769,0.406613,0.184654,0.233339,0.128646,0.048721,0.049412,0.046241,0.123311,0.410404,0.122464,0.049412,0.646901,0.080507,0.046241,0.592510,0.154072,0.162055,0.146337,0.261810,0.066849,0.126103,0.219463,0.048721,0.175746,0.680614,0.146705,0.046241,0.048721,0.087937,0.049412,0.436563,0.930093,0.122464,0.403684,0.086830,0.249742,0.046241,0.048721,0.130925,0.720357,0.122464,0.046241,0.259759,0.823944,0.591142,0.048721,0.221907,0.046241,0.049412,0.288032,0.049412,0.067108,0.231423,0.461633
3160,0.732100,0.784577,0.485936,0.952046,NaN,0.207297,0.624877,0.841212,0.954425,0.678947,0.947705,0.847195,0.855276,0.753096,0.293468,0.293947,0.237558,0.727069,0.952046,0.733634,0.687173,0.952046,0.560237,0.432551,0.952046,0.754215,0.847610,0.558231,0.736994,0.626629,0.732951,0.840697,0.258638,0.729111,0.952046,0.752518,0.352917,0.264478,0.727289,0.448352,0.927230,0.954425,0.626507,0.948283,0.738950,0.840250,0.120860,0.411077,0.749523,0.952477,0.621114,0.226295,0.841929,0.954425,0.933643,0.144748,0.823731,0.185768,0.537521,0.818101,0.467328,0.697403,0.856663,0.952046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3435,0.141203,0.229648,0.224651,0.706538,0.188281,0.046241,0.244465,0.472856,0.847963,0.255883,0.697634,0.815763,0.305128,0.245257,0.219806,0.211803,0.048517,0.440324,0.710830,0.249244,0.209592,0.842184,0.219379,0.100723,0.843911,0.269654,0.580988,0.234767,0.272700,0.244151,0.145962,0.256010,0.105598,0.248600,0.860205,0.258281,0.082846,0.154011,0.248600,0.088667,0.783782,0.936798,0.226752,0.844337,0.310672,0.421809,0.035916,0.053723,0.261458,0.862417,0

In [63]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Alabama,Arizona,Baylor,Chattanooga,Colorado,Connecticut,Creighton,Drake,Drexel,Duke,E Washington,Fairfield,FGCU,Florida St,Gonzaga,Indiana,Iowa,Iowa St,Jackson St,Kansas,Kansas St,Kent,Louisville,LSU,Maine,Marquette,Marshall,Maryland,Michigan,Michigan St,Mississippi,MTSU,NC State,Nebraska,Norfolk St,North Carolina,Notre Dame,Ohio St,Oklahoma,Oregon St,Portland,Presbyterian,Princeton,Rice,Richmond,S Dakota St,South Carolina,Stanford,Syracuse,TAM C. Christi,Tennessee,Texas,Texas A&M,TN Martin,UC Irvine,UCLA,UNLV,USC,Utah,Vanderbilt,Virginia Tech,West Virginia,WI Green Bay,Cal Baptist
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Alabama,NaN,0.464361,0.284176,0.872115,0.263652,0.059661,0.251614,0.731626,0.939002,0.377849,0.776080,0.737840,0.720819,0.722212,0.144686,0.226917,0.137388,0.297087,0.870120,0.528151,0.372310,0.872463,0.453128,0.135520,0.872892,0.705978,0.846284,0.295208,0.629402,0.272700,0.243216,0.685219,0.245352,0.334641,0.871724,0.682506,0.227643,0.227916,0.436957,0.268284,0.871659,0.939055,0.333246,0.859669,0.685231,0.727081,0.042903,0.120129,0.386821,0.939055,0.181534,0.120505,0.829489,0.939002,0.871620,0.132429,0.424698,0.133570,0.234763,0.855036,0.246814,0.281530,0.610610,0.871759
Arizona,0.522571,NaN,0.267489,0.847610,0.213671,0.060299,0.346719,0.726952,0.934227,0.494232,0.845713,0.737094,0.614547,0.630604,0.158258,0.233352,0.149048,0.323161,0.871833,0.588827,0.452987,0.871669,0.251191,0.211457,0.873541,0.703333,0.787576,0.299431,0.714680,0.364824,0.354099,0.608364,0.240150,0.407208,0.872995,0.674202,0.255122,0.173633,0.412189,0.184859,0.870120,0.939055,0.315787,0.870807,0.655863,0.726513,0.046241,0.094121,0.616908,0.933700,0.296615,0.063569,0.707178,0.939055,0.872866,0.070368,0.330899,0.177490,0.515349,0.762757,0.246755,0.405455,0.658308,0.847610
Baylor,0.714035,0.732431,NaN,0.952046,0.510266,0.233476,0.601510,0.764878,0.954425,0.713842,0.947969,0.846558,0.763309,0.761441,0.440570,0.295652,0.334055,0.398347,0.952477,0.554387,0.438702,0.954425,0.607347,0.406126,0.952046,0.765060,0.871395,0.632906,0.744336,0.619730,0.753734,0.720774,0.290954,0.698982,0.954425,0.734705,0.419855,0.372071,0.524189,0.500162,0.939034,0.954425,0.627557,0.954425,0.678179,0.764325,0.133366,0.239718,0.702704,0.954425,0.610417,0.207089,0.767530,0.954425,0.934285,0.325262,0.647834,0.423429,0.682581,0.772434,0.501746,0.775772,0.812979,0.952046
Chattanooga,0.123920,0.146705,0.049412,NaN,0.049412,0.046241,0.080969,0.248213,0.549876,0.066769,0.406613,0.184654,0.233339,0.128646,0.048721,0.049412,0.046241,0.123311,0.410404,0.122464,0.049412,0.646901,0.080507,0.046241,0.592510,0.154072,0.162055,0.146337,0.261810,0.066849,0.126103,0.219463,0.048721,0.175746,0.680614,0.146705,0.046241,0.048721,0.087937,0.049412,0.436563,0.930093,0.122464,0.403684,0.086830,0.249742,0.046241,0.048721,0.130925,0.720357,0.122464,0.046241,0.259759,0.823944,0.591142,0.048721,0.221907,0.046241,0.049412,0.288032,0.049412,0.067108,0.231423,0.461633
Colorado,0.732100,0.784577,0.485936,0.952046,NaN,0.207297,0.624877,0.841212,0.954425,0.678947,0.947705,0.847195,0.855276,0.753096,0.293468,0.293947,0.237558,0.727069,0.952046,0.733634,0.687173,0.952046,0.560237,0.432551,0.952046,0.754215,0.847610,0.558231,0.736994,0.626629,0.732951,0.840697,0.258638,0.729111,0.952046,0.752518,0.352917,0.264478,0.727289,0.448352,0.927230,0.954425,0.626507,0.948283,0.738950,0.840250,0.120860,0.411077,0.749523,0.952477,0.621114,0.226295,0.841929,0.954425,0.933643,0.144748,0.823731,0.185768,0.537521,0.818101,0.467328,0.697403,0.856663,0.952046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vanderbilt,0.141203,0.229648,0.224651,0.706538,0.188281,0.046241,0.244465,0.472856,0.847963,0.255883,0.697634,0.815763,0.305128,0.245257,0.219806,0.211803,0.048517,0.44

In [64]:
df_matrix.to_csv(f'../data/simulations/womens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/simulations/womens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'